In [ ]:
# ============================================================
# DEEP_DIVE_YUKAWA_V2: 4 ТОЧКИ (1 МЛН ШАГОВ) С ПОЛНЫМИ ЧЕКПОИНТАМИ
# ============================================================

# REM: Импорт библиотеки numpy для работы с массивами и математическими функциями на CPU.
import numpy as np

# REM: Импорт функции label из scipy для поиска связных кластеров (4-связность) на CPU.
from scipy.ndimage import label

# REM: Импорт библиотеки matplotlib для отрисовки графиков.
import matplotlib

# REM: Включаем фоновый режим 'Agg', чтобы графики сохранялись в файлы, а не открывались в окнах.
matplotlib.use('Agg') 

# REM: Импорт модуля pyplot для создания картинок.
import matplotlib.pyplot as plt

# REM: Импорт модуля csv для записи таблиц с результатами.
import csv

# REM: Импорт модуля time для замера времени выполнения расчетов.
import time

# REM: Импорт модуля os для работы с файловой системой (создание папок, проверка путей).
import os

# REM: Импорт функции display и класса Image для вывода картинок прямо в Jupyter.
from IPython.display import display, Image as IPImage

# ------------------------------------------------------------
# REM: ПАПКИ ДЛЯ РЕЗУЛЬТАТОВ И ЧЕКПОИНТОВ
# ------------------------------------------------------------
# REM: Фиксированное имя папки. Благодаря этому при перезапуске кода он найдет старые чекпоинты.
output_dir = "DEEP_DIVE_YUKAWA_V2_5_billion"

# REM: Путь к подпапке для CSV-файлов (таблиц).
csv_dir = os.path.join(output_dir, "csv")

# REM: Путь к подпапке для картинок (PNG).
png_dir = os.path.join(output_dir, "png")

# REM: Путь к подпапке для чекпоинтов (сохранений сетки).
ckpt_dir = os.path.join(output_dir, "checkpoints")

# REM: Создаем все папки. exist_ok=True предотвращает ошибку, если папка уже существует.
os.makedirs(output_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)
os.makedirs(png_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

# ------------------------------------------------------------
# REM: ФИЗИЧЕСКИЕ ПАРАМЕТРЫ
# ------------------------------------------------------------
# REM: L - размер решетки (100x100 узлов). Всего 10 000 плексонов.
L = 100

# REM: STEPS - количество итераций внешнего цикла.
# REM: ВАЖНО: Внутри цикла стоит `for _ in range(5)`, поэтому реальное число шагов = 1 000 000 * 5 = 5 000 000.
STEPS = 1_000_000  

# REM: beta - обратная температура (1/kT). Отвечает за уровень случайного шума (термофлуктуации).
beta = 1.0

# REM: J0 - сила притяжения (амплитуда потенциала Юкавы).
J0 = 3.0

# REM: mu - параметр экранирования. Определяет, как быстро сила затухает с расстоянием.
mu = 0.5

# REM: r_cutoff - радиус обрезания. Взаимодействие считается только на расстоянии до 4 клеток.
r_cutoff = 4

# REM: Список точек для расчета: кортежи вида (Плотность, Сигма).
# REM: (0.10, 2.0) - 10% плотность, Сигма 2.0 (бинодальная зона, капли).
# REM: (0.10, 10.0) - 10% плотность, Сигма 10.0 (сильное натяжение).
# REM: (0.50, 1.5) - 50% плотность, Сигма 1.5 (спинодальная зона, гель).
# REM: (0.50, 2.0) - 50% плотность, Сигма 2.0 (стабильный гель).
POINTS = [
    (0.10, 2.0),   
    (0.10, 10.0),  
    (0.50, 1.5),   
    (0.50, 2.0)    
]

# REM: Формируем полный путь к CSV-файлу.
csv_file = os.path.join(csv_dir, "DEEP_DIVE_YUKAWA_V2_5_billion.csv")

# REM: Если файла еще нет, создаем его и записываем заголовки колонок.
if not os.path.exists(csv_file):
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['p_init', 'J0', 'sigma', 'step', 'density', 'P_inf', 'perimeter', 'max_size'])

# ------------------------------------------------------------
# REM: ВСПОМОГАТЕЛЬНЫЕ И ФИЗИЧЕСКИЕ ФУНКЦИИ
# ------------------------------------------------------------

def init_lattice(L, p):
    # REM: Создает случайную решетку. С вероятностью p узел занят (1), иначе пуст (0).
    return (np.random.random((L, L)) < p).astype(int)

def make_yukawa_kernel(r_cut, mu, J0_val):
    # REM: Создает 2D-матрицу (ядро) сил Юкавы.
    # REM: Размер матрицы 9x9 (от -4 до +4).
    size = 2 * r_cut + 1
    kernel = np.zeros((size, size))
    
    # REM: Заполняем ядро по формуле: J0 * exp(-mu * r) / r.
    for i in range(size):
        for j in range(size):
            dx, dy = i - r_cut, j - r_cut
            r = np.sqrt(dx**2 + dy**2)
            # REM: В центре (r=0) силы нет, иначе будет деление на ноль.
            if r > 0 and r <= r_cut:
                kernel[i, j] = J0_val * np.exp(-mu * r) / r
    return kernel

def get_local_window_torus(grid, i, j, r_cut):
    # REM: Вырезает окно (9x9) вокруг узла (i, j).
    # REM: Благодаря операции % L (остаток от деления), границы зациклены (тор).
    L = grid.shape[0]
    window = np.zeros((2*r_cut+1, 2*r_cut+1))
    for di in range(-r_cut, r_cut+1):
        for dj in range(-r_cut, r_cut+1):
            window[di + r_cut, dj + r_cut] = grid[(i + di) % L, (j + dj) % L]
    return window

def local_energy(grid, i, j, kernel, r_cut):
    # REM: Считает энергию взаимодействия ОДНОГО узла с его окружением.
    # REM: Мы берем окно вокруг узла, умножаем на ядро и суммируем.
    # REM: Знак минус означает, что взаимодействие притягивает (снижает энергию).
    win = get_local_window_torus(grid, i, j, r_cut)
    return -np.sum(win * kernel)

def get_local_perimeter_both(grid, i1, j1, i2, j2, radius=2):
    # REM: Считает изменение периметра только в локальной области вокруг двух изменяемых узлов.
    # REM: Это в разы быстрее, чем считать периметр всей решетки на каждом шаге.
    L = grid.shape[0]
    p = 0
    # REM: Определяем границы окна вокруг узлов (i1, j1) и (i2, j2).
    min_i, max_i = min(i1, i2) - radius, max(i1, i2) + radius
    min_j, max_j = min(j1, j2) - radius, max(j1, j2) + radius
    
    for i in range(min_i, max_i + 1):
        for j in range(min_j, max_j + 1):
            ni, nj = i % L, j % L
            if grid[ni, nj] == 1:
                # REM: Если сосед пуст (0), то это граница (периметр +1).
                if grid[(ni-1)%L, nj] == 0: p += 1
                if grid[(ni+1)%L, nj] == 0: p += 1
                if grid[ni, (nj-1)%L] == 0: p += 1
                if grid[ni, (nj+1)%L] == 0: p += 1
    return p

def metropolis_step_yukawa(grid, sigma, yukawa_kernel, beta=1.0):
    # REM: Один шаг алгоритма Метрополиса (обмен местами занятого и пустого узла).
    L = grid.shape[0]
    
    # REM: Ищем случайный занятый узел (1).
    while True:
        i1, j1 = np.random.randint(0, L, 2)
        if grid[i1, j1] == 1: break
        
    # REM: Ищем случайный пустой узел (0).
    while True:
        i2, j2 = np.random.randint(0, L, 2)
        if grid[i2, j2] == 0: break

    # REM: Считаем энергию Юкавы ДО обмена (только для i1, так как i2 пуст и энергии не дает).
    E_old = local_energy(grid, i1, j1, yukawa_kernel, r_cutoff)
    
    # REM: Считаем локальный периметр ДО обмена.
    P_old = get_local_perimeter_both(grid, i1, j1, i2, j2)

    # REM: Временно меняем узлы местами (1 -> 0, 0 -> 1).
    grid[i1, j1] = 0
    grid[i2, j2] = 1

    # REM: Считаем энергию Юкавы ПОСЛЕ обмена (только для i2, так как i1 теперь пуст).
    E_new = local_energy(grid, i2, j2, yukawa_kernel, r_cutoff)
    
    # REM: Считаем локальный периметр ПОСЛЕ обмена.
    P_new = get_local_perimeter_both(grid, i1, j1, i2, j2)

    # REM: Полное изменение энергии: (Разница энергий Юкавы) + Сигма * (Разница периметров).
    delta_S = (E_new - E_old) + sigma * (P_new - P_old)

    # REM: Критерий Метрополиса: принимаем, если энергия упала или по вероятности Больцмана.
    if delta_S <= 0 or np.random.random() < np.exp(-beta * delta_S):
        return True
    else:
        # REM: Если не приняли, возвращаем узлы на место.
        grid[i1, j1] = 1
        grid[i2, j2] = 0
        return False

def measure(grid):
    # REM: Функция для измерения характеристик системы.
    # REM: np.pad с mode='wrap' добавляет рамку, чтобы учесть зацикливание при поиске кластеров.
    padded = np.pad(grid, pad_width=1, mode='wrap')
    
    # REM: Ищем кластеры 4-связности.
    labeled, n = label(padded, structure=[[0,1,0],[1,1,1],[0,1,0]])
    
    # REM: Убираем добавленную рамку.
    labeled = labeled[1:-1, 1:-1]
    
    # REM: Считаем размеры кластеров.
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    sizes = sizes[sizes > 0]
    
    # REM: Если кластеров нет, возвращаем None.
    if len(sizes) == 0: return None
    
    # REM: P_inf - доля узлов в самом большом кластере.
    P_inf = float(sizes.max() / np.sum(sizes))
    
    # REM: Реальная плотность.
    density = float(np.mean(grid))
    
    # REM: Считаем глобальный периметр (через сдвиг матриц).
    right = np.roll(grid, 1, axis=1); left = np.roll(grid, -1, axis=1)
    up = np.roll(grid, 1, axis=0); down = np.roll(grid, -1, axis=0)
    perimeter = int(np.sum((grid == 1) & (right == 0)) + np.sum((grid == 1) & (left == 0)) + 
                    np.sum((grid == 1) & (up == 0)) + np.sum((grid == 1) & (down == 0)))
    
    return {'density': density, 'P_inf': P_inf, 'perimeter': perimeter, 'max_size': int(sizes.max())}

# ------------------------------------------------------------
# REM: ЗАПУСК С ПОДДЕРЖКОЙ ВОССТАНОВЛЕНИЯ (ЧЕКПОИНТЫ)
# ------------------------------------------------------------

# REM: Создаем ядро Юкавы один раз перед запуском.
yukawa_kernel = make_yukawa_kernel(r_cutoff, mu, J0)

# REM: Начинаем перебор по всем 4 точкам.
for p, sigma in POINTS:
    # REM: Формируем пути к файлам чекпоинтов для текущей точки.
    ckpt_grid_file = os.path.join(ckpt_dir, f"grid_p{p}_s{sigma}.npy")
    ckpt_step_file = os.path.join(ckpt_dir, f"step_p{p}_s{sigma}.npy")
    
    # REM: Начальный шаг 0, сетка пока не создана.
    start_step = 0
    grid = None
    
    # REM: Проверяем наличие сохраненной сетки и шага (если расчет прерывался).
    if os.path.exists(ckpt_grid_file) and os.path.exists(ckpt_step_file):
        grid = np.load(ckpt_grid_file)
        start_step = int(np.load(ckpt_step_file)[0])
        print(f"\n[RESTART] Плотность {p*100}%, Сигма {sigma}. Восстанавливаем с шага {start_step*5}...", flush=True)
    
    # REM: Если сохранений нет, проверяем, не завершена ли точка в CSV.
    elif os.path.exists(csv_file):
        is_done = False
        with open(csv_file, 'r') as f:
            reader = csv.reader(f)
            next(reader, None) # Пропускаем заголовок
            for row in reader:
                if len(row) >= 4 and abs(float(row[0]) - p) < 1e-5 and abs(float(row[2]) - sigma) < 1e-5:
                    if int(row[3]) == STEPS * 5: is_done = True
        
        if is_done:
            print(f"[SKIP] Плотность {p*100}%, Сигма {sigma} уже полностью посчитана.", flush=True)
            continue # Переходим к следующей точке
        else:
            grid = init_lattice(L, p)
    else:
        grid = init_lattice(L, p)

    # REM: Печатаем заголовок таблицы для текущей точки.
    print("\n" + "="*65, flush=True)
    print(f"СЧИТАЕМ (Юкава): Плотность = {p*100}%, Сигма = {sigma}, J0 = {J0}", flush=True)
    print("="*65, flush=True)
    print(f"{'Шаг':>8} | {'density':>8} | {'P_inf':>8} | {'Perimeter':>10} | {'Max_Size':>8} | {'Time(s)':>7}", flush=True)
    print("-" * 65, flush=True)

    # REM: Засекаем время начала расчета текущей точки.
    t0 = time.time()
    
    # REM: Основной цикл. Если было восстановление, начинаем с start_step.
    for step in range(start_step, STEPS):
        # REM: Внутренний цикл делает 5 шагов Метрополиса за 1 итерацию.
        for _ in range(5):
            metropolis_step_yukawa(grid, sigma, yukawa_kernel, beta)
        
        # REM: Вычисляем реальное количество сделанных шагов.
        real_step = step * 5
        
        # REM: Каждые 250 000 реальных шагов сохраняем результат и картинку.
        if real_step % 250_000 == 0 and real_step > 0:
            # REM: Делаем замер характеристик.
            m = measure(grid)
            
            # REM: Считаем затраченное время.
            elapsed = time.time() - t0
            
            # REM: Записываем строку в CSV.
            with open(csv_file, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([p, J0, sigma, real_step, m['density'], m['P_inf'], m['perimeter'], m['max_size']])
            
            # REM: Печатаем строку в консоль.
            print(f"{real_step:>8} | {m['density']:>8.3f} | {m['P_inf']:>8.3f} | {m['perimeter']:>10.1f} | {m['max_size']:>8d} | {elapsed:>7.1f}", flush=True)
            
            # REM: Сохраняем текущее состояние сетки и шага на диск.
            np.save(ckpt_grid_file, grid)
            np.save(ckpt_step_file, np.array([step]))
            
            # REM: Создаем картинку текущего состояния.
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(grid, cmap='binary', interpolation='nearest')
            ax.set_title(f'Yukawa D={p*100}%, σ={sigma}, Step={real_step}')
            
            # REM: Сохраняем картинку в файл.
            img_path = os.path.join(png_dir, f"Y_D{p}_S{sigma}_step{real_step}.png")
            plt.savefig(img_path, dpi=100, bbox_inches='tight')
            plt.close(fig)
            
            # REM: Выводим картинку в Jupyter.
            display(IPImage(filename=img_path))

    # REM: После завершения цикла делаем финальный замер.
    m = measure(grid)
    
    # REM: Записываем финальную точку в CSV.
    with open(csv_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([p, J0, sigma, STEPS*5, m['density'], m['P_inf'], m['perimeter'], m['max_size']])
    
    # REM: Удаляем файлы чекпоинтов, так как расчет полностью завершен.
    if os.path.exists(ckpt_grid_file): os.remove(ckpt_grid_file)
    if os.path.exists(ckpt_step_file): os.remove(ckpt_step_file)
    
    print(f"ТОЧКА ЗАВЕРШЕНА.", flush=True)

print("\nВСЕ 4 ТОЧКИ ЮКАВЫ ГОТОВЫ! Данные в DEEP_DIVE_YUKAWA_V2_5_billion.", flush=True)